# M7 Evaluation: Retrieval + Generation

This notebook documents the complete M7 evaluation workflow for EvidenceRAG on the QASPER validation split.

It is intentionally distributed **without execution outputs**. Running the notebook generates a fresh experiment under `experiments/m7/`. The project's reference benchmark is preserved separately under `reference_results/` and is never overwritten.

## 1. What M7 evaluates

M7 evaluates four retrieval configurations using the standardized M4/M5 settings:

- **BM25**
- **Dense retrieval** using Qwen3-Embedding-0.6B
- **Hybrid** retrieval using reciprocal-rank fusion
- **Hybrid + Reranker** using the M5 cross-encoder

Retrieval metrics include Recall@5, Recall@20, MRR@5, and Evidence F1.

End-to-end generation additionally evaluates QASPER Answer F1 and Answer F1 by answer type.

The validation split contains **1,005 questions**, producing **4,020 system-level records** because each question is evaluated by all four systems.

## 2. Environment setup

In [ ]:
!pip install -e .

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))

## 3. Verify M2/M3 inputs

M7 consumes the processed QASPER data produced by M2 and M3.

It does not silently regenerate missing preprocessing artifacts.

In [ ]:
from pathlib import Path

required = [
    Path("data/processed/validation.jsonl"),
    Path("data/processed/chunks/validation.jsonl"),
    Path("data/processed/chunks/validation_evidence_map.jsonl"),
]

for path in required:
    status = "OK" if path.exists() else "MISSING"
    print(f"{path}: {status}")

## 4. Full retrieval evaluation

Run the standardized retrieval benchmark over all 1,005 validation questions.

This creates a new runtime result directory under `experiments/m7/`.

Use a new `--run-id` for a new experiment. Reusing the same run ID resumes an interrupted run.

In [ ]:
!python scripts/evaluate_m7.py --split validation --mode retrieval --run-id validation-retrieval

## 5. Generation without re-running retrieval

Generation can be performed from a completed retrieval run.

This is the recommended M7 path because the expensive corpus embedding/retrieval stage does not need to be repeated.

`generate_answers_m7.py` reads the already-produced `retrieved_chunk_ids`, reconstructs the M6 prompts, generates answers, and computes Answer F1.

In [ ]:
# Replace validation-retrieval with the retrieval run you want to use.

!python scripts/generate_answers_m7.py --split validation --source-run-id validation-retrieval --run-id validation-e2e-full --max-new-tokens 64 --batch-size 1

## 6. Optional two-GPU generation

For a full validation generation run, records can be divided across two GPUs.

Each process writes to a distinct run directory, avoiding concurrent writes to the same JSONL file.

In [ ]:
# GPU 0
# CUDA_VISIBLE_DEVICES=0 python scripts/generate_answers_m7.py --split validation --source-run-id validation-retrieval --run-id validation-gen-gpu0 --max-new-tokens 64 --paper-start 0 --paper-end 141

# GPU 1
# CUDA_VISIBLE_DEVICES=1 python scripts/generate_answers_m7.py --split validation --source-run-id validation-retrieval --run-id validation-gen-gpu1 --max-new-tokens 64 --paper-start 141 --paper-end 281

## 7. Resumability

M7 writes records incrementally.

If a run is interrupted, rerunning it with the same `--run-id` resumes from completed records rather than regenerating them.

Keeping retrieval and generation as separate stages also prevents the expensive embedding build from being repeated after an interruption.

## 8. Reference benchmark

This repository includes the completed project benchmark under:

`reference_results/m7_validation/`

These files are **reference artifacts only**.

They are not used as the output directory of a fresh experiment. New evaluations write to `experiments/m7/<run-id>/`, which is intentionally ignored by Git.

In [ ]:
import json
from pathlib import Path

summary_path = Path("reference_results/m7_validation/summary.json")
summary = json.loads(summary_path.read_text())

for system in summary["systems"]:
    print(
        system["system"],
        "Answer F1:",
        system["answer_f1"],
    )

## 9. Final reported validation benchmark

The reference run contains 1,005 questions for each of the four systems.

The final Answer F1 values are recorded in `summary.json` and can be compared against a fresh reproduction run.

## 10. Reproducibility notes

- The notebook is distributed without execution outputs.
- Runtime experiment outputs belong under `experiments/m7/`.
- Reference benchmark artifacts belong under `reference_results/m7_validation/`.
- Reference results are never overwritten by reproduction runs.
- The reported validation benchmark uses `max_new_tokens=64`.
- Generation can be resumed using the same run ID.
- Two independent GPU processes can be used when additional GPU capacity is available.